In [2]:
import snowflake.connector
from sqlalchemy import create_engine
from sqlalchemy.dialects import registry
import re
import pandas as pd
import deepl

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/snowflake/connector/options.py:104: UserWarning: You have an incompatible version of 'pyarrow' installed (19.0.1), please install a version that adheres to: 'pyarrow<19.0.0; extra == "pandas"'
  warn_incompatible_dep(


In [3]:
# for installing dependencies

credentials = {
    'account': 'uniper.west-europe.azure',
    'user': 'M02555@uniper.energy',
    'authenticator': 'externalbrowser',
    'role': 'PRD_COODE_HSSE_ANALYST',
    'warehouse': 'PRD_WH_UGC_COODE_QUERY'
}

conn = snowflake.connector.connect(**credentials)

In [4]:
df = pd.read_csv('RW_ACTUALS_ALL_05_05.csv')

/tmp/ipykernel_3165/951073567.py:1: DtypeWarning: Columns (38) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('RW_ACTUALS_ALL_05_05.csv')


In [5]:
df

,CASENO,COMPANY,FUNCTIONAL_GROUP,FUNCTION,FUNCTIONAL_AREA,FUNCTIONAL_LOCATION,FUNCTIONAL_SUB_LOCATION,LOCATION_SID,LOCATION_SHORT,COUNTRY_SHORT,...,PERSONAL_INJURIES,CASE_CLOSED_DATE,CASES_NO_OF_REGISTRATIONS,VALID_FROM,VALID_TO,POTENTIAL_SEV_LEVEL,RISK_AREA,LEARNINGS_ACTUAL_SEVERITY,LEARNINGS_POTENTIAL_RISK,COMPANY_NAME
0,20494,Uniper Group (70006651),COO (70017898),Asset Operations (70017927),Steam and Biomass (70017928),Karlshamnsverket SE (70018695),NaN,NaN,NaN,NaN,...,NaN,2022-09-14,1.0,2025-04-28 16:04:37.000,2100-12-31 00:00:00.000,Very low,Low Risk,Low,Low,NaN
1,26763,Uniper Group (70006651),COO (70017898),Energy Assets (75000907),Flexible Energy GT (75000929),Karlshamn (75001383),NaN,NaN,NaN,NaN,...,NaN,2022-09-14,1.0,2025-04-28 16:04:37.000,2100-12-31 00:00:00.000,Very low,Low Risk,Low,Low,NaN
2,32999,Uniper Group (70006651),COO (70017898),Asset Operations (70017927),Steam and Biomass (70017928),Karlshamnsverket SE (70018695),NaN,NaN,NaN,NaN,...,NaN,2023-05-07,1.0,2025-04-28 16:04:37.000,2100-12-31 00:00:00.000,Very low,Low Risk,Low,Low,NaN
3,38121,Uniper Group (70006651),COO (70017898),Asset Operations (70017927),Steam and Biomass (70017928),Karlshamnsverket SE (70018695),NaN,NaN,NaN,NaN,...,NaN,2024-01-08,1.0,2025-04-28 16:04:37.000,2100-12-31 00:00:00.000,Low,Low Risk,Low,Low,NaN
4,28188,Uniper Group (70006651),COO (70017898),Asset Operations (70017927),Steam and Biomass (70017928),Karlshamnsverket SE (70018695),NaN,NaN,NaN,NaN,...,NaN,2022-10-05,1.0,2025-04-28 16:04:37.000,2100-12-31 00:00:00.000,Very low,Low Risk,Low,Low,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23544,28185,Uniper Group (70006651) - -- Not selected --,COO (70017898),Asset Operations (70017927),Steam and Biomass (70017928),Karlshamnsverket SE (70018695),NaN,NaN,NaN,NaN,...,NaN,2022-09-14,1.0,2025-04-28 16:04:37.000,2100-12-31 00:00:00.000,Not selected,Not selected,Not Applicable,Low,NaN
23545,28184,Uniper Group (70006651) - Uniper,COO (70017898),Asset Operations (70017927),Steam and Biomass (70017928),Karlshamnsverket SE (70018695),NaN,NaN,NaN,NaN,...,NaN,2022-09-14,1.0,2025-04-28 16:04:37.000,2100-12-31 00:00:00.000,Not selected,Not selected,Not Applicable,Low,NaN
23546,40163,Uniper Group (70006651) - -- Not selected --,COO (70017898),Asset Operations (70017927),CCGT (70017930),City Plants (70019214),FFM-PLS-OPE (70019042),NaN,NaN,NaN,...,NaN,2025-03-28,1.0,2025-04-28 16:04:37.000,2100-12-31 00:00:00.000,Very low,Low Risk,Low,Low,NaN
23547,53480,Uniper Group (70006651) - -- Not selected --,COO (70017898),Energy Assets (75000907),Flexible Energy GT (75000929),Grain UK (75001395),Production Team (75001616),8138.0,PP GRA,UK,...,NaN,2025-04-30,1.0,2025-04-30 12:05:08.000,2100-12-31 00:00:00.000,Low,Low Risk,Low,Low,NaN


9e126190-00a7-4eee-ad78-756b6d7b117b

In [7]:
# Set up DeepL with your API key
DeepL_API_Key = "9e126190-00a7-4eee-ad78-756b6d7b117b"
translator = deepl.Translator(DeepL_API_Key)

# Specify only the two columns to translate
columns_to_translate = ['TITLE', 'CASE_DESCRIPTION', 'IMM_ACTION_TAKEN_RECOM']
print(f"Columns to translate: {columns_to_translate}")

# Create a new DataFrame for the translated data
df_translated = df.copy()

# Get indices of rows where COUNTRY_SHORT is not UK
non_uk_indices = df_translated[df_translated['COUNTRY_SHORT'] != 'UK'].index

print(f"Found {len(non_uk_indices)} non-UK rows to process")

# Translate only the specified columns for non-UK rows
for column in columns_to_translate:
    print(f"Translating column: {column} for non-UK rows")
    
    # Process only non-UK rows
    for idx in non_uk_indices:
        value = df_translated.at[idx, column]
        if isinstance(value, str) and value.strip():
            try:
                df_translated.at[idx, column] = translator.translate_text(value, target_lang="EN-GB").text
            except Exception as e:
                print(f"Error translating row {idx}, column {column}: {e}")
    
print("Translation completed - result stored in df_translated DataFrame")

# Show the first few rows of the translated DataFrame
print(df_translated.head())

# Save the translated DataFrame to a CSV file
output_filename = 'RW_ACTUALS_ALL_05_TRANSLATED.csv'
df_translated.to_csv(output_filename, index=False)
print(f"Translated data saved to {output_filename}")

Columns to translate: ['TITLE', 'CASE_DESCRIPTION', 'IMM_ACTION_TAKEN_RECOM']
Found 21096 non-UK rows to process
Translating column: TITLE for non-UK rows
Translating column: CASE_DESCRIPTION for non-UK rows
Translating column: IMM_ACTION_TAKEN_RECOM for non-UK rows
Translation completed - result stored in df_translated DataFrame
   CASENO                  COMPANY FUNCTIONAL_GROUP  \
0   20494  Uniper Group (70006651)   COO (70017898)   
1   26763  Uniper Group (70006651)   COO (70017898)   
2   32999  Uniper Group (70006651)   COO (70017898)   
3   38121  Uniper Group (70006651)   COO (70017898)   
4   28188  Uniper Group (70006651)   COO (70017898)   

                      FUNCTION                FUNCTIONAL_AREA  \
0  Asset Operations (70017927)   Steam and Biomass (70017928)   
1     Energy Assets (75000907)  Flexible Energy GT (75000929)   
2  Asset Operations (70017927)   Steam and Biomass (70017928)   
3  Asset Operations (70017927)   Steam and Biomass (70017928)   
4  Asset Ope

In [ ]:
df = pd.read_csv('RW_ACTUALS_ALL_05_TRANSLATED.csv')

In [ ]:
df = df_translated.drop_duplicates(subset='CASE_DESCRIPTION').reset_index()

In [ ]:
from bertopic import BERTopic
 
docs = df.TITLE_EN.astype(str) +'. ' +  list(df.CASE_DESCRIPTION_EN) # +' This took place in ' + df.SL_COUNTRY.astype(str))

docs[0:10]

In [ ]:
#Oskarshamn 3 - RA3-23 ERF - 
#[re.sub('Oskarshamn \d - [A-Z0-9]+ - ','', a) for a in docs if re.search('Oskarshamn \d - [A-Z0-9]+ - ', a)]

#docs = list(df.CASE_DESCRIPTION_EN)
docs = [re.sub('Oskarshamn \d - [A-Z0-9]+ - ','', case) for case in docs]
docs = [re.sub('Oskarshamn \d - ','', case) for case in docs]
docs = [re.sub('Oskarshamn \d','', case) for case in docs]
docs = [re.sub('Risk less:','', case) for case in docs]
docs[0:10]

In [ ]:
from sentence_transformers import SentenceTransformer
sentence_model = SentenceTransformer("all-mpnet-base-v2", device = 'cuda') #all-mpnet-base-v2 all-MiniLM-L6-v2

embeddings = sentence_model.encode(docs, show_progress_bar=True)


In [ ]:
import nltk
nltk.download('wordnet')

In [ ]:
from bertopic.representation import PartOfSpeech, KeyBERTInspired, MaximalMarginalRelevance
from sklearn.feature_extraction.text import CountVectorizer
from hdbscan import HDBSCAN
from umap import UMAP
from bertopic.dimensionality import BaseDimensionalityReduction

pos_patterns = [
            [{'POS': 'ADJ'}, {'POS': 'NOUN'}],
            [{'POS': 'NOUN'}], [{'POS': 'ADJ'}]
]
pos_patterns = [
            [{'POS': 'NOUN'}]]
#representation_model = PartOfSpeech("en_core_web_sm", pos_patterns=pos_patterns)
#representation_model = KeyBERTInspired()
representation_model = MaximalMarginalRelevance(diversity=0.3)

#representation_model = [PartOfSpeech("en_core_web_sm"), KeyBERTInspired(), MaximalMarginalRelevance(diversity=0.3)]
#representation_model = MaximalMarginalRelevance(diversity=0.3)


# The main representation of a topic
#main_representation = PartOfSpeech("en_core_web_sm", pos_patterns=pos_patterns)

# Additional ways of representing a topic
aspect_model1 = MaximalMarginalRelevance(diversity=0.5)

# Add all models together to be run in a single `fit`
#representation_model = {
#   "Main": main_representation,
#   "Aspect1":  aspect_model1,
#}

#representation_model = [ PartOfSpeech("en_core_web_sm", pos_patterns=pos_patterns), MaximalMarginalRelevance(diversity=.5)]

representation_model = KeyBERTInspired()
#topic_model = BERTopic(representation_model=representation_model).fit(docs)


vectorizer_model = CountVectorizer(stop_words="english")
from nltk import word_tokenize          
from nltk.stem import WordNetLemmatizer 

class LemmaTokenizer:
    def __init__(self):
        self.wnl = WordNetLemmatizer()
    def __call__(self, doc):
        return [self.wnl.lemmatize(t) for t in word_tokenize(doc)]

vectorizer_model= CountVectorizer(tokenizer=LemmaTokenizer()) 
hdbscan_model = HDBSCAN(min_cluster_size=5, min_samples =5, metric='euclidean', cluster_selection_method='eom', prediction_data=True) # #
umap_model = UMAP(n_neighbors=8, n_components=20, metric='cosine', low_memory=False)
#umap_model = BaseDimensionalityReduction()

topic_model = BERTopic(
    min_topic_size= 7,
   # n_gram_range = (1,2),
    vectorizer_model=vectorizer_model,
    #hdbscan_model=hdbscan_model,
    #umap_model = umap_model,
    #nr_topics=100,
    calculate_probabilities=True,
    embedding_model=sentence_model,
    representation_model=representation_model).fit(docs, embeddings )

In [ ]:
df_embeddings = pd.DataFrame(embeddings)
df_embeddings['CASENO'] = df.CASENO

In [ ]:
df_embeddings

In [ ]:
topics, probs = topic_model.fit_transform(docs, embeddings)

In [ ]:
topic_model.get_topic_info()

In [ ]:
labels = [topic_model.get_topic(topic)[0] for topic in sorted(list(set(topic_model.topics_)))]
#labels

In [ ]:
# Reduce outliers
topic_reduced = topic_model.reduce_outliers(docs, topics, probabilities=probs, strategy="probabilities", threshold= 0.01)
#topic_reduced = topic_model.reduce_outliers(docs, topics, probabilities=probs)
#topic_reduced = topic_model.reduce_outliers(docs, topics, strategy="embeddings", threshold=0.01)
#topic_reduced = topic_model.reduce_outliers(docs, topics , strategy="c-tf-idf", threshold=0.1)

topics = topic_reduced
topic_model.update_topics(docs, topics=topic_reduced)

In [ ]:
topic_model.get_topic_info()

In [ ]:
len([a for a in topic_reduced if a<0])

In [ ]:
df_topic_labels = pd.DataFrame(topic_model.topic_labels_.items())
df_topic_labels

In [ ]:
df

In [ ]:
df_topics_intermed = df_topics_intermed.merge(df_topic_labels, how="left", on= 0)
df_topics_intermed.columns = [0,1,'label']

df_topics_intermed= df_topics_intermed.explode(1)
df_topics_intermed['word'], df_topics_intermed['value'] = zip(*df_topics_intermed[1])

In [ ]:
df_topics = df_topics_intermed.iloc[:,[0,2,3,4]]
df_topics

In [ ]:
from umap import UMAP
import numpy as np
import torch
embeddings = df_embeddings.values[: ,0:-1].astype(np.float32)
embeddings_casenos = df_embeddings.values[:, -1].astype(int)
embeddings_tensor = torch.tensor(embeddings)
embeddings_2d = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)

In [ ]:
import pickle

with open('embeddings_2d.pkl', 'wb') as f:
    pickle.dump(embeddings_2d, f)

In [ ]:
df['topic'] = topics
df[['CASENO', 'topic']]
df.reset_index(drop=True, inplace=True)
df

In [ ]:
import pickle

with open('synergi_tm_results.pkl', 'wb') as f:
    pickle.dump([df_embeddings, df_topics, df], f)

In [ ]:
topic_model.visualize_topics()

In [ ]:
from umap import UMAP

In [ ]:

# Run the visualization with the original embeddings
topic_model.visualize_documents(docs, embeddings=embeddings)

# Reduce dimensionality of embeddings, this step is optional but much faster to perform iteratively:
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
topic_model.visualize_documents(docs, reduced_embeddings=reduced_embeddings)

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
hierarchical_topics = topic_model.hierarchical_topics(docs)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)


In [ ]:
fig = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)
fig.write_html("topics_hierarchy.html")

In [ ]:
# Run the visualization with the original embeddings
topic_model.visualize_hierarchical_documents(docs, hierarchical_topics, embeddings=embeddings)

# Reduce dimensionality of embeddings, this step is optional but much faster to perform iteratively:
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
topic_model.visualize_hierarchical_documents(docs, hierarchical_topics, reduced_embeddings=reduced_embeddings)

In [ ]:
topic_model.visualize_barchart(top_n_topics=8, n_words=8)

In [ ]:
topic_model.topic_representations_

In [ ]:
len([a for a in topics if a ==-1])

In [ ]:
[(a,b) for a,b in zip(docs, topics) if b == -1]

In [ ]:
topic_model.visualize_heatmap()

In [ ]:
timestamps = df.CASE_OCCURENCE_DATE

In [ ]:
topics_over_time = topic_model.topics_over_time(docs=docs,timestamps=timestamps)

In [ ]:
topic_model.visualize_topics_over_time(topics_over_time, topics=list(range(0,20)))

In [ ]:
classes = df.FUNCTION

In [ ]:
topics_per_class = topic_model.topics_per_class(docs, classes=classes)

In [ ]:
topic_model.visualize_topics_per_class(topics_per_class)

In [ ]:
topic_distr, _ = topic_model.approximate_distribution(docs, min_similarity=0)

In [ ]:
# To visualize the probabilities of topic assignment
topic_model.visualize_distribution(probs[5])


In [ ]:
# To visualize the topic distributions in a document
topic_model.visualize_distribution(topic_distr[0])

In [ ]:
# Calculate the topic distributions on a token-level
topic_distr, topic_token_distr = topic_model.approximate_distribution(docs, calculate_tokens=True)

# Visualize the token-level distributions
df_tl = topic_model.visualize_approximate_distribution(docs[1], topic_token_distr[1])
df_tl